### 1. Write a program to make a statistical estimate for π using one of the two methods suggested in ‘Maths13’

I will estimate $\pi$ using a Monte-Carlo simualtion of a $[1,1]$ square.

In [ ]:
import numpy as np

seed = np.random.default_rng()

def estimate_pi(N, r=1.0):
    x = seed.random(N)
    y = seed.random(N)
    inside = np.count_nonzero(x**2 + y**2 < r**2)
    return 4 / r**2 * inside / N


for N in [10**2, 10**4, 10**6, 10**8]:
    pi_hat = estimate_pi(N)
    print(f"N = {N}: pi_hat = {pi_hat:.5f}, error = {pi_hat - np.pi:+.5f}")

### 2. How accurate is the Monte Carlo?

In [ ]:
"""Task 2: repeat the pi estimate many times, fit the spread, check 1.64/sqrt(N)."""
import os

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

M = 2000  # number of repeats for each N
os.makedirs("out", exist_ok=True)

# --- (a) histogram of M estimates at one N, with a Gaussian fit ---------------
N = 10_000
pi_hats = np.array([estimate_pi(N) for _ in range(M)])
mu, sigma = norm.fit(pi_hats)  # best-fit Gaussian (mean, std)
p = np.pi / 4
print(f"N = {N}: fitted mean {mu:.4f}, fitted sigma {sigma:.4f}, "
      f"theory 4*sqrt(p(1-p)/N) = {4 * np.sqrt(p * (1 - p) / N):.4f}")

# pi_hat only takes values 4m/N, so put bin edges half-way between allowed values
step = 4 / N
m_lo, m_hi = round(pi_hats.min() / step), round(pi_hats.max() / step)
edges = step * (np.arange(m_lo, m_hi + 10, 8) - 0.5)  # 8 allowed values per bin

xs = np.linspace(pi_hats.min(), pi_hats.max(), 300)
plt.hist(pi_hats, bins=edges, density=True, alpha=0.5, label=f"{M} runs")
plt.plot(xs, norm.pdf(xs, mu, sigma), label="Gaussian fit")
plt.axvline(np.pi, color="k", ls="--", label=r"$\pi$")
plt.xlabel(r"$\hat\pi$"); plt.ylabel("density"); plt.legend()
plt.savefig("out/pi_histogram.png", dpi=150); plt.close()

# --- (b) how the spread scales with N ------------------------------------------
Ns = np.array([100, 300, 1000, 3000, 10_000, 30_000])
sigmas = np.array([np.std([estimate_pi(n) for _ in range(M)]) for n in Ns])
slope, _ = np.polyfit(np.log(Ns), np.log(sigmas), 1)
print(f"slope of log(sigma) vs log(N) = {slope:.3f} (theory -0.5)")

plt.loglog(Ns, sigmas, "o", label="measured spread")
plt.loglog(Ns, 4 * np.sqrt(p * (1 - p) / Ns), label=r"$4\sqrt{p(1-p)/N}$")
plt.xlabel("$N$"); plt.ylabel(r"$\sigma_{\hat\pi}$"); plt.legend()
plt.savefig("out/pi_scaling.png", dpi=150); plt.close()

# --- (c) r = 1 versus the handout's r^2 = 2/3 -----------------------------------
for r2 in [1.0, 2 / 3]:
    rel = np.std([estimate_pi(N, np.sqrt(r2)) for _ in range(M)]) / np.pi
    p_r = np.pi * r2 / 4
    print(f"r^2 = {r2:.3f}: relative error * sqrt(N) = {rel * np.sqrt(N):.3f}, "
          f"theory sqrt((1-p)/p) = {np.sqrt((1 - p_r) / p_r):.3f}")
